# Fine-Tune a Specialist Adapter (book-scale edition)

Companion notebook to *Build AI, Understand AI*, Part VI, Chapter 2.
Distilled from the LocoLabo capstone training notebook.

You will teach a small base model a dialect: a voice, format, and set
of domain habits, by training a LoRA adapter on examples of good
answers, then measure it against the base model on a sealed eval set.

**Simplified, not production.** Needs a Colab GPU (Runtime > T4).
At book scale (50-100 examples) expect tens of minutes, not the
capstone's hours. A negative result, rigorously shown, is success.


## Step 0: Confirm you have a GPU

If this errors or shows no GPU, fix the runtime type before going further.

In [ ]:
!nvidia-smi


## Step 1: Install Unsloth

Unsloth makes LoRA training fit in a free Colab GPU. It may ask you to restart the session; if so, restart and run from here.

In [ ]:
!pip install -q unsloth unsloth_zoo


## Step 2: Settings

Defaults match the LocoLabo capstone so results stay comparable.
Change one at a time and log what you changed.


In [ ]:
ADAPTER_NAME = "plain-english"   # lowercase, no spaces

BASE_MODEL = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 1024

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LEARNING_RATE = 2e-4
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4

QUANT_METHOD = "q4_k_m"
SEED = 42


## Step 3: Training data (a runnable miniature)

The format is JSONL: one `{"conversations": [user, assistant]}` record
per line. The cell below builds a tiny built-in dataset teaching one
dialect (short plain-English answers) so the whole pipeline can be
smoke-tested. For your real run, replace it with 50-100 of your own
examples and keep a separate sealed eval file that never trains.


In [ ]:
import json

style_qa = [
    ("What does the adapter learn in fine-tuning?",
     "It learns habits from examples: voice, format, and what a good answer looks like here. It does not reliably learn new facts."),
    ("Why keep an eval set sealed?",
     "If test examples leak into training, scores flatter you and mean nothing. Sealed means the model never trains on them."),
    ("What is LoRA in one sentence?",
     "LoRA freezes the base model and trains a small extra layer beside it, well under one percent of the weights."),
    ("When should I use retrieval instead of fine-tuning?",
     "When the gap is facts that change or must be cited. Retrieval supplies evidence at question time; fine-tuning cannot."),
]
records = [
    {"conversations": [
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ]}
    for q, a in style_qa
]
# Repeat with light variation is a capstone job; book scale starts here.
records = records * 15  # 60 examples: enough to smoke-test, not to publish
eval_records = [
    {"conversations": [
        {"role": "user", "content": "Does fine-tuning replace documentation search?"},
        {"role": "assistant", "content": "No. It changes how answers sound and behave. Current facts still need retrieval at question time."},
    ]},
]
with open("training_data.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")
with open("eval_dataset.jsonl", "w") as f:
    for r in eval_records:
        f.write(json.dumps(r) + "\n")
print(len(records), "training records;", len(eval_records), "sealed eval records (grow this)")


## Step 4: Check the data before training on it

Format errors cost seconds here and an afternoon after training.
Also check for train/eval overlap (leakage) explicitly.


In [ ]:
problems = []
seen = set()
for n, rec in enumerate(records, 1):
    convo = rec.get("conversations")
    if not isinstance(convo, list) or len(convo) < 2:
        problems.append(f"record {n}: needs at least 2 turns"); continue
    if convo[0].get("role") != "user" or convo[1].get("role") != "assistant":
        problems.append(f"record {n}: first turns must be user then assistant"); continue
    if not str(convo[1].get("content", "")).strip():
        problems.append(f"record {n}: empty assistant answer"); continue
    key = convo[0]["content"].strip().lower()
    if key in seen:
        problems.append(f"record {n}: duplicate prompt")
    seen.add(key)
eval_prompts = {r["conversations"][0]["content"].strip().lower() for r in eval_records}
leaks = seen & eval_prompts
print("problems:", problems or "none")
print("leaked eval prompts:", leaks or "none")
for rec in records[:2]:
    print("USER:     ", rec["conversations"][0]["content"][:160])
    print("ASSISTANT:", rec["conversations"][1]["content"][:160])


## Step 5: Load the base model (the one you must beat)

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)


## Step 6: Add the LoRA specialisation layer

The printout is the claim under test: a tiny fraction of trainable weights, aimed well, beating the general model at one dialect.

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")


## Step 7: Format examples with the model's chat template

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(records)

def format_conversations(examples):
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(format_conversations, batched=True)
print(dataset[0]["text"][:600])


## Step 8: Train (SFT)

Watch the loss fall and flatten. Near-zero loss on a tiny dataset means
memorisation, and your sealed eval, not this number, decides success.
Log the final loss and every setting change, including failures.


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="checkpoints",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        fp16=True,
        logging_steps=5,
        save_strategy="epoch",
        seed=SEED,
        optim="adamw_8bit",
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)
stats = trainer.train()
print(stats)


## Step 9: Export a merged GGUF for Ollama

Merges the adapter into the base and quantises to about 2.5 GB. Disk-hungry; copy the result out (Drive) before closing the tab — Colab deletes everything at session end.

In [ ]:
OUTPUT_DIR = f"{ADAPTER_NAME}_gguf"
model.save_pretrained_gguf(OUTPUT_DIR, tokenizer, quantization_method=QUANT_METHOD)
import os
for f in os.listdir(OUTPUT_DIR):
    if f.endswith(".gguf"):
        print(f, f"({os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024**3:.1f} GB)")


## Step 10: Judge base versus adapter on the sealed set

Back on your laptop, register both models in Ollama (base, and the
exported GGUF via a one-line Modelfile: `FROM ./<file>.gguf`), then ask
every sealed eval question of both and mark them blind with Chapter 11's
judge plus your own reading. Report both scores, whichever way it lands.

## Optional extension: preferences (DPO), the simple RL step

Generate two answers per eval-style prompt with the SFT model, rank them
with your Chapter 11 criteria (human marks first, judge second), and
save `{"prompt", "chosen", "rejected"}` pairs. A DPO pass (TRL's
`DPOTrainer`, same LoRA settings, far fewer steps) then nudges weights
toward chosen over rejected. GRPO goes further and only earns its keep
where a Python reward function can verify answers (format, arithmetic,
tests passing). Neither belongs in a first run; both are judged by the
same sealed set, untouched throughout.
